### **Import libraries**: 

In [25]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import sys
import fastparquet
import traceback

### **Load Parquet file into a Pandas DataFrame object and calculate mean sentiment, sentiment standard deviation, number of articles, number of strongly negative articles, and sentiment momentum.**

In [26]:
def load_and_calculate(parquet_file_name_param, all_metadata):
    df = pd.read_parquet(parquet_file_name_param)
    tickers = list(set(df.index.get_level_values(0)))
    tickers.sort()
    dates = list(set(df.index.get_level_values(1)))
    dates = [str(element) for element in dates]
    dates.sort()
    headlineIDs = list(set(df.index.get_level_values(2)))
    headlineIDs.sort()
    iterables = [tickers, pd.to_datetime(dates)]
    index = pd.MultiIndex.from_product(iterables, names=["Ticker", "Timestamp"])
    df_aggregated = pd.DataFrame(index = index, columns = ["sent_mean", "sent_std", "news_count", "pct_strong_negative", "sent_momentum"])
    df_aggregated.sort_index(inplace = True, level = 1, ascending = False, sort_remaining = False)

    counter = 0
    total = len(tickers) * len(dates)


    print(dates[10:])

    # tickers = ["AMD"]

    # df_aggregated = df_aggregated.loc[["AMD", "AMZN"]]

    # print(df)

    # print(dates[:10])
    

    # #print(df.loc["AMD"])

    # #print(df.loc["AMD", "2021-04-23 08:00:0", ])

    # #print(df_aggregated.loc["AMD", ])

    # #sys.exit("exit")
    
    for ticker in tickers:
        for date, headlineID in zip(dates, headlineIDs):
            try:
                sent_diff = df.loc[ticker, date, headlineID]["pos"].sub(df.loc[ticker, date, headlineID]["neg"], axis = 0)
                sent_mean_today = sent_diff.mean()
                sent_std_today = sent_diff.std()
                news_count_today = len(sent_diff)
                if news_count_today == 1:
                    sent_std_today = 0
                pct_strong_negative_today = (len(df.loc[ticker, date, headlineID]["neg"][df.loc[ticker, date, headlineID]["neg"]>0.7]) / len(df.loc[ticker, date, headlineID]["neg"]))*100
                try:
                    sent_5_day_avg = np.mean([(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i), headlineID]["pos"].sub(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i), headlineID]["neg"], axis = 0)).mean() for i in range(5)])
                    sent_momentum = sent_mean_today - sent_5_day_avg
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, sent_momentum
                    
                except KeyError:
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, np.nan
                
            except KeyError:
                pass
                #traceback.print_exc()
                #sys.exit()
                #print("No article was published about " + str(ticker) + " today.")
        
            # with open("status.txt", "w") as f:
            #     f.writelines(all_metadata)
            #     f.write(str(round((counter/total)*100, 2)) + "% complete." + "\n")

            counter +=1

    df_aggregated[df_aggregated.columns] = df_aggregated[df_aggregated.columns].apply(pd.to_numeric, axis = 1)
    return df_aggregated
    #df_aggregated.index.set_levels(new_time_index level=1, inplace=True)


### **Main program used to collectively execute the whole script.**

In [27]:
def main():

    # with open("status.txt", "r+") as f:
    #     content = f.read()
    #     time_now = datetime.now()
    #     start_string = "Phase 3 started on " + time_now.strftime("%d-%m-%Y") + " at " + time_now.strftime("%H:%M:%S") + "." + "\n"
    #     f.write(start_string)
    #     f.seek(0)
    #     all_metadata = f.readlines()
    
    parquet_file_name = "5_year_ticker_headlines_with_finbert_rating.parquet"
    df_with_alt_data_features = load_and_calculate(parquet_file_name, "")

    print(df_with_alt_data_features)
    
    #df_with_alt_data_features.to_parquet("5_year_ticker_headlines_with_finbert_rating_calculated.parquet")

    # with open("status.txt", "w") as f:
    #     f.writelines(all_metadata)
    #     time_now = datetime.now()
    #     end_string = "Phase 3 completed on " + time_now.strftime("%d-%m-%Y") + " at " + time_now.strftime("%H:%M:%S") + "." + "\n"
    #     f.write(end_string) 
        
main()

                                                                                Headline  \
Ticker Timestamp           HeadlineID                                                      
JPM    2026-04-23 23:39:59 f2e2a2b8    Amer Sports Rose 10% Over the Past 30 Days. He...   
NKE    2026-04-23 23:39:59 df227634    Amer Sports Rose 10% Over the Past 30 Days. He...   
       2026-04-23 23:39:22 4070b740    |‘Iconic’: Columbia marks 30 years of the Baha...   
AAPL   2026-04-23 23:26:37 6c76c88e    Hasbro preliminary sales beat estimates, buoye...   
NKE    2026-04-23 23:26:37 de24b347    Hasbro preliminary sales beat estimates, buoye...   
...                                                                                  ...   
COP    2021-04-23 13:00:00 81673802    Development plans for Troll West electrificati...   
MSFT   2021-04-23 11:15:57 cfad1c4d    Evercore joins as advisor for S.Korea’s Hanon ...   
TSLA   2021-04-23 11:05:00 60ca2d7d    Is MicroVision (MVIS) Really the Great Bu